In [1]:
!nvidia-smi

Tue Apr 29 04:26:03 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [18]:
!wget O embedding_layer.npz "https://github.com/Subrat1920/IMDB-Sentiment-Analysis/raw/refs/heads/main/Notebook1/sources/embeddings_compressed.npz"
!wget O embedding_model.keras "https://github.com/Subrat1920/IMDB-Sentiment-Analysis/raw/refs/heads/main/Notebook1/sources/embedding_model.keras"
!wget -O dataset.csv 'https://github.com/Subrat1920/IMDB-Sentiment-Analysis/raw/refs/heads/main/Notebook/Datasets/cleaned_sentiment_data.csv'

--2025-04-29 04:35:00--  http://o/
Resolving o (o)... failed: Name or service not known.
wget: unable to resolve host address ‘o’
--2025-04-29 04:35:00--  http://embedding_layer.npz/
Resolving embedding_layer.npz (embedding_layer.npz)... failed: Name or service not known.
wget: unable to resolve host address ‘embedding_layer.npz’
--2025-04-29 04:35:00--  https://github.com/Subrat1920/IMDB-Sentiment-Analysis/raw/refs/heads/main/Notebook1/sources/embeddings_compressed.npz
Resolving github.com (github.com)... 140.82.116.4
Connecting to github.com (github.com)|140.82.116.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://media.githubusercontent.com/media/Subrat1920/IMDB-Sentiment-Analysis/refs/heads/main/Notebook1/sources/embeddings_compressed.npz [following]
--2025-04-29 04:35:00--  https://media.githubusercontent.com/media/Subrat1920/IMDB-Sentiment-Analysis/refs/heads/main/Notebook1/sources/embeddings_compressed.npz
Resolving media.githubusercontent.

In [5]:
import tensorflow as tf
embedding_model = tf.keras.models.load_model("embedding_model.keras")
embedding_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1137)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 1137, 10)       │       155,170 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 155,170 (606.13 KB)

 Trainable params: 155,170 (606.13 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
import numpy as np
embeddings = np.load('embeddings_compressed.npz')

In [14]:
list(embeddings.files)

['embeddings']

In [15]:
embeddings = embeddings['embeddings']

In [16]:
embeddings.shape

(50000, 1137, 10)

In [19]:
import pandas as pd
df = pd.read_csv('dataset.csv')
df.shape

(50000, 3)

In [20]:
df.head(3)

,Unnamed: 0,Reviews,Sentiments
0,0,this film was just brilliant casting location ...,1
1,1,big hair big boobs bad music and a giant safet...,0
2,2,this has to be one of the worst films of the 1...,0


In [29]:
sentiments = df.Sentiments.tolist()
sentiments[:5]

[1, 0, 0, 1, 0]

In [30]:
print('Embeddings for the first sentence')
print('-'*100)
print(embeddings[0])
print('-'*100)
print('Sentiment for the first sentence')
print('-'*100)
print(sentiments[0])

Embeddings for the first sentence
----------------------------------------------------------------------------------------------------
[[ 0.0361364  -0.02924824  0.01106571 ...  0.02482151 -0.01247789
   0.00587802]
 [ 0.02586483 -0.01904371  0.04512474 ... -0.00806425 -0.01032783
   0.02636118]
 [ 0.01346442 -0.03212177 -0.03362333 ... -0.04709277 -0.0351317
  -0.01510924]
 ...
 [-0.03115041 -0.01955576  0.01819123 ...  0.01559296  0.02607734
  -0.00958972]
 [-0.03115041 -0.01955576  0.01819123 ...  0.01559296  0.02607734
  -0.00958972]
 [-0.03115041 -0.01955576  0.01819123 ...  0.01559296  0.02607734
  -0.00958972]]
----------------------------------------------------------------------------------------------------
Sentiment for the first sentence
----------------------------------------------------------------------------------------------------
1


In [33]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(embeddings, sentiments, test_size=0.2, random_state=42)

In [65]:
x_train = np.array(x_train)
x_test = np.array(x_test)
y_train = np.array(y_train)
y_test = np.array(y_test)

In [54]:
max_len = x_train.shape[1]
vocabulary_size = 15517
embedding_dim = 10
print(f'So, we have now maximum length of a review: {max_len} with logged vocabulary size of {vocabulary_size}, and we have embedding dimention of {embedding_dim}')

So, we have now maximum length of a review: 1137 with logged vocabulary size of 15517, and we have embedding dimention of 10


## This is where we will do the model building, and if not goes well drop all of them below this, until furthure instruction

In [52]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GRU, Bidirectional, Embedding, Dropout, Input
import warnings
warnings.filterwarnings('ignore')

In [58]:
model = Sequential()
model.add(Bidirectional(GRU(128, return_sequences=True), input_shape=(max_len, embedding_dim)))
model.add(Dropout(0.3))
model.add(GRU(64))
model.add(Dropout(0.3))
model.add(Dense(1, activation='sigmoid'))

In [59]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional_3 (Bidirectional) │ (None, 1137, 256)      │       107,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 1137, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_7 (GRU)                     │ (None, 64)             │        61,824 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 169,409 (661.75 KB)

 Trainable params: 169,409 (661.75 KB)

 Non-trainable params: 0 (0.00 B)

In [60]:
from tensorflow.keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

In [62]:
opt = tf.keras.optimizers.Adam(learning_rate=0.0005)
loss = tf.keras.losses.BinaryCrossentropy()

In [63]:
model.compile(optimizer=opt, loss=loss, metrics=['accuracy'])

In [66]:
history = model.fit(
    x_train, y_train,
    epochs=20,
    batch_size=64,
    validation_data=(x_test, y_test),
    callbacks=[early_stopping]
)

Epoch 1/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 91s 133ms/step - accuracy: 0.5003 - loss: 0.6935 - val_accuracy: 0.5088 - val_loss: 0.6930
Epoch 2/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 137s 131ms/step - accuracy: 0.4939 - loss: 0.6936 - val_accuracy: 0.4912 - val_loss: 0.6937
Epoch 3/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 84s 135ms/step - accuracy: 0.5022 - loss: 0.6935 - val_accuracy: 0.4912 - val_loss: 0.6933
Epoch 4/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 142s 135ms/step - accuracy: 0.5030 - loss: 0.6933 - val_accuracy: 0.4912 - val_loss: 0.6932


In [80]:
import nltk, re, string
from nltk.corpus import stopwords
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
nltk.download('stopwords')
nltk.download('puckt')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Error loading puckt: Package 'puckt' not found in index
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [74]:
stop_words = set(stopwords.words('english'))
def clean_text(text):
  text = text.lower()
  text = re.sub(f"[{string.punctuation}]", "", text)
  text = re.sub(r'\d+', '', text)
  text = ' '.join([word for word in text.split() if word not in stop_words])
  return text

In [81]:
tokenizer = Tokenizer(num_words=vocabulary_size+1, oov_token="<OOV>")
def predict_sentiment(text):
    text = clean_text(text)
    seq = tokenizer.texts_to_sequences([text])
    padded_seq = pad_sequences(seq, maxlen=max_len, padding='post')
    result = model.predict(padded_seq)
    return result


In [82]:
example1 = 'The movie was very good'
predict_sentiment(example1)

TypeError: int() argument must be a string, a bytes-like object or a real number, not 'NoneType'